# Final Load Preparation


In [1]:
%pip install pandas numpy gdown

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Load Cleaned Dataset


In [2]:
import gdown
df = pd.read_csv(gdown.download(id='1a-MNVGzc5DV-MTDmkWOCP5qhVTaaZMnu', quiet=False))

# Ensure date is parsed correctly
# Convert the 'order_date' column from string text into actual datetime objects for time-based analysis
df['order_date'] = pd.to_datetime(df['order_date'])
df.head()


,order_id,order_date,product_id,store_id,customer_id,quantity,unit_price,discount,revenue,cost,...,weight_g,store_name,city,country,store_type,year,month,day,week,day_of_week
0,0RD00000001,2023-01-07,P0080,S093,C040749,5,14.43,0.15,61.33,42.77,...,200.0,Chocolate Store 93,Sydney,UK,Airport,2023,1,7,1,5
1,0RD00000002,2023-10-22,P0173,S065,C020161,3,12.01,0.00,36.03,19.06,...,50.0,Chocolate Store 65,New York,Australia,Retail,2023,10,22,42,6
2,0RD00000003,2023-05-07,P0115,S078,C048069,2,10.02,0.00,20.04,10.29,...,50.0,Chocolate Store 78,London,UK,Airport,2023,5,7,18,6
3,0RD00000004,2024-06-23,P0186,S088,C047901,2,14.66,0.10,26.39,16.35,...,50.0,Chocolate Store 88,Toronto,USA,Retail,2024,6,23,25,6
4,0RD00000005,2024-09-24,P0197,S054,C033950,1,12.34,0.00,12.34,7.94,...,120.0,Chocolate Store 54,London,Canada,Online,2024,9,24,39,1


## 2. Feature Engineering for BI Tools
We will create some derived features that make building dashboards in Tableau much easier.


In [3]:
# 1. Age Groups
bins = [0, 25, 35, 45, 60, 100]
labels = ['18-25', '26-35', '36-45', '46-60', '60+']
# Bin the continuous 'age' column into discrete groups (e.g., '18-25', '26-35') using predefined bins
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=False)

# 2. Profit Margin
# Calculate the profit margin as a percentage of total revenue
df['profit_margin'] = df['profit'] / df['revenue']

# 3. Discount Flag
# Create a boolean flag (True/False) indicating whether a discount was applied to the order
df['has_discount'] = df['discount'] > 0

# 4. Date Components
# Extract the quarter from the order date and format it as 'Q1', 'Q2', etc.
df['quarter'] = 'Q' + df['order_date'].dt.quarter.astype(str)

df[['age', 'age_group', 'revenue', 'profit', 'profit_margin', 'discount', 'has_discount', 'order_date', 'quarter']].head()


,age,age_group,revenue,profit,profit_margin,discount,has_discount,order_date,quarter
0,44,36-45,61.33,18.56,0.302625,0.15,True,2023-01-07,Q1
1,63,60+,36.03,16.97,0.470996,0.00,False,2023-10-22,Q4
2,35,36-45,20.04,9.75,0.486527,0.00,False,2023-05-07,Q2
3,37,36-45,26.39,10.04,0.380447,0.10,True,2024-06-23,Q2
4,57,46-60,12.34,4.40,0.356564,0.00,False,2024-09-24,Q3


## 3. Data Type Optimization
To save memory and speed up load times, we will downcast numerical types and convert categorical text columns to `category`.


In [4]:
# Downcast numerics
# Downcast 64-bit integers to 32-bit or 16-bit integers to save memory
df['quantity'] = df['quantity'].astype('int32')
df['age'] = df['age'].astype('int16')

# Categorize strings
cat_cols = ['category', 'store_type', 'brand', 'gender', 'city', 'country', 'age_group', 'quarter', 'loyalty_member', 'has_discount']
for col in cat_cols:
    # Convert string columns with few unique values into 'category' type to significantly reduce memory usage
    df[col] = df[col].astype('category')

# Print a concise summary of the DataFrame, including data types and memory usage
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 33 columns):
 #   Column          Non-Null Count    Dtype         
---  ------          --------------    -----         
 0   order_id        1000000 non-null  str           
 1   order_date      1000000 non-null  datetime64[us]
 2   product_id      1000000 non-null  str           
 3   store_id        1000000 non-null  str           
 4   customer_id     1000000 non-null  str           
 5   quantity        1000000 non-null  int32         
 6   unit_price      1000000 non-null  float64       
 7   discount        1000000 non-null  float64       
 8   revenue         1000000 non-null  float64       
 9   cost            1000000 non-null  float64       
 10  profit          1000000 non-null  float64       
 11  age             1000000 non-null  int16         
 12  gender          1000000 non-null  category      
 13  loyalty_member  1000000 non-null  category      
 14  join_date       1000000 non-nu

## 4. Export Final BI-Ready Dataset
We will save the optimized dataset as a CSV file in the `processed` directory. You can import this file directly into Tableau.


In [5]:
output_path = '../data/processed/BI_Ready_Dataset.csv'
# Export the fully processed and enriched DataFrame to a CSV file without the index column
df.to_csv(output_path, index=False)
print(f'Saved BI-Ready dataset to {output_path}')


Saved BI-Ready dataset to ../data/processed/BI_Ready_Dataset.csv
